# 04_reflection_and_rewrite

05_reflection_and_rewrite.py — Self-Reflection + Query Rewriting

(1) 충실성 검사: 답변이 검색 문서에 근거하는가? (환각 여부)
(2) 유용성 검사: 답변이 질문에 실제로 답하는가?
(3) 쿼리 재작성: 검색에 더 적합하도록 질문을 다듬는다.

In [1]:
import os, sys, ssl, certifi
# Windows 인증서 저장소 손상 우회(임베딩/HTTPS 로드 SSL 에러 방지)
ssl.SSLContext.load_default_certs = lambda self, *a, **k: self.load_verify_locations(certifi.where())
import nest_asyncio; nest_asyncio.apply()
# 노트북 커널엔 __file__ 이 없으므로 스크립트 호환 위해 정의 + supp/ 를 import 경로에 추가
__file__ = os.path.join(os.getcwd(), '04_reflection_and_rewrite.py')
sys.path.insert(0, os.path.abspath('..'))

In [2]:
"""
05_reflection_and_rewrite.py — Self-Reflection + Query Rewriting

(1) 충실성 검사: 답변이 검색 문서에 근거하는가? (환각 여부)
(2) 유용성 검사: 답변이 질문에 실제로 답하는가?
(3) 쿼리 재작성: 검색에 더 적합하도록 질문을 다듬는다.
"""
import sys as _sys
from pathlib import Path as _Path
_sys.path.insert(0, str(_Path(__file__).resolve().parent.parent))

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from _common import get_llm, binary_yesno, banner, llm_unavailable


HALLUCINATION_SYSTEM = (
    "너는 답변이 주어진 문서에 근거하는지 평가한다. "
    "답변 안의 사실이 문서로부터 도출 가능하면 yes, 지어낸 것이면 no."
)
ANSWER_SYSTEM = "답변이 사용자 질문의 의도를 만족하면 yes, 아니면 no."

REWRITE_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "주어진 질문을 벡터 검색에 더 적합하도록 한 줄로 재작성하라. "
     "숨은 의미를 드러내고 핵심 키워드를 명확히 하라. "
     "결과만 출력 (설명 X)."),
    ("human", "원본 질문: {question}"),
])


def main() -> None:
    llm = get_llm()
    if llm is None:
        llm_unavailable()
        return

    question_rewriter = REWRITE_PROMPT | llm | StrOutputParser()

    docs = (
        "LLM 에이전트의 메모리는 단기 메모리(컨텍스트 윈도우)와 "
        "장기 메모리(벡터DB 검색)로 나뉜다."
    )
    question = "에이전트 메모리에는 어떤 종류가 있나?"

    grounded_answer = "에이전트 메모리는 단기 메모리와 장기 메모리로 나뉜다."
    hallucinated_answer = "에이전트 메모리는 RAM 16GB 와 SSD 1TB 로 구성된다."

    banner("Reflection #1 — 충실성 (groundedness)")
    for label, ans in [("근거 있음", grounded_answer), ("환각", hallucinated_answer)]:
        score = binary_yesno(
            llm, HALLUCINATION_SYSTEM,
            f"문서:\n{docs}\n\n답변:\n{ans}",
        )
        print(f"  [{label:>6}]  '{ans}' → 근거={score}")

    banner("Reflection #2 — 유용성 (usefulness)")
    irrelevant_answer = "오늘 날씨가 좋다."
    for label, ans in [("관련 답변", grounded_answer), ("무관 답변", irrelevant_answer)]:
        score = binary_yesno(
            llm, ANSWER_SYSTEM,
            f"질문:\n{question}\n\n답변:\n{ans}",
        )
        print(f"  [{label:>8}]  '{ans}' → 유용={score}")

    banner("Iterative Retrieval — 쿼리 재작성")
    vague = "그거 어떻게 해?"
    better = question_rewriter.invoke({"question": vague})
    print(f"  원본:    {vague!r}")
    print(f"  재작성:  {better.strip()}")


if __name__ == "__main__":
    main()

D:\git\2604_agent_210h_handson\supp\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



📌 Reflection #1 — 충실성 (groundedness)


  [ 근거 있음]  '에이전트 메모리는 단기 메모리와 장기 메모리로 나뉜다.' → 근거=yes


  [    환각]  '에이전트 메모리는 RAM 16GB 와 SSD 1TB 로 구성된다.' → 근거=no

📌 Reflection #2 — 유용성 (usefulness)


  [   관련 답변]  '에이전트 메모리는 단기 메모리와 장기 메모리로 나뉜다.' → 유용=yes


  [   무관 답변]  '오늘 날씨가 좋다.' → 유용=no

📌 Iterative Retrieval — 쿼리 재작성


  원본:    '그거 어떻게 해?'
  재작성:  그거 수행 방법은?
